# 02 The universe and its returns, checked against Stock-Level Tracking

**What this notebook does.** It builds the universe, the S&P 500's members month by month, and their monthly returns with delistings applied, with the code copied from Stock-Level Tracking. It then checks that the copy gives Stock-Level Tracking's committed counts to the last digit, and counts the member-months of the panel's window.

**Why it comes here.** The copied data layer was built and tested in Stock-Level Tracking, against CRSP's own count of members and its own index returns. Reproducing that study's committed tables here shows that the copy, run on this machine's extraction through this study's wrapper, is the same layer. Everything after this notebook rests on it.

**What it judges.** Gate G1 (a). If it is not met, nothing after this notebook runs.

**The data.** CRSP, from Wharton Research Data Services, accessed through Queen Mary University of London, extracted on 28 September 2026.

## Terms used in this notebook

| Term | Meaning |
|---|---|
| universe | the stocks in CRSP's list of S&P 500 members on the last trading day of the month before. A stock is in the universe in month t when one of its membership spells covers the last trading day of month t-1 |
| member-month | one stock in the universe in one month |
| membership spell | one interval in CRSP's list, from the day a stock entered the index to the day it left, both days included |
| PERMNO | CRSP's permanent number for a security. It does not change when the company changes its name or its ticker |
| CRSP's count | the number of securities in the index on a day, which CRSP stores on each row of its monthly index file |
| count check | the universe's count in each month beside CRSP's count on the same day, summarised by year: the mean difference and the months more than 10 names apart |
| delisting | the end of a stock's listing on its exchange. CRSP records its date, a code for its reason and, where it can, a delisting return |
| delisting rule | Stock-Level Tracking's rule for a delisting, after Shumway (1997): CRSP's delisting return where it gives one, minus 30 per cent for a code from 400 to 599 where it gives none, and zero for any other code. In a month with both a return and a delisting return, the two are compounded |
| total return | the month's return with dividends and with the delisting rule applied |
| start-of-month capitalisation | the absolute price times shares outstanding at the end of month t-1, in thousands of dollars |
| reference table | one of the three tables Stock-Level Tracking committed at the commit the copied code is taken from, read from GitHub by that commit's hash |
| gate G1 (a) | the copied layer gives the reference tables' member-months, count check by year and delisting counts by year to the last digit |
| panel's window | the months January 1991 to December 2024, the months of the study's panel. The test window, January 2006 to December 2024, is its last 228 months |

## What I expect to see, and where it comes from

1. CRSP's list holds 2,064 spells of 1,936 PERMNOs, and the universe runs 1,188 months, January 1926 to December 2024, with 440,724 member-months: Stock-Level Tracking's `universe_summary.json`, at the commit in the constants.
2. The count check by year, 99 rows, is the same file byte for byte as Stock-Level Tracking's `universe_count_check_by_year.csv`.
3. The delisting counts by year, 99 rows, are the same file byte for byte as Stock-Level Tracking's `delistings_by_year.csv`.
4. Over the panel's window: 204,520 member-months, of which 100 have no return and no delisting, and 114,521 over the test window. These are the constants of section 3, read from the same reference table on 18 November 2026. They are compared here and judged by no gate.

Expectations 1 to 3 together are gate G1 (a). Running time: about a minute.

In [ ]:
import hashlib
import json
import pathlib
import sys
import time

# The package lives in src/ beside this folder. The notebook finds the repository's root from where it runs.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "mlrp").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from mlrp import constants as C
from mlrp import data as D
from mlrp.source import extraction as X
from mlrp.source import layer as L
from mlrp.source import returns as R
from mlrp.source import universe as U

# Stops with one sentence when the data root is not set, when the manifest is not the one the constants fix,
# or when one of the study's eight tables is not the one they record.
root, manifest = D.open_extraction()
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}", "| entries of the eight tables", C.MANIFEST_ENTRIES_SHA256[:12])

## 1. CRSP's list and the universe

The list is read whole: PERMNO, start and end. Five counts say whether it is well formed: spells, PERMNOs, PERMNOs with more than one spell, spells that overlap an earlier spell of the same PERMNO, and spells that end before they start. The universe is then built month by month from the list and from CRSP's month-end dates in its monthly index file.

In [ ]:
t0 = time.time()
universe = L.load_universe(root)
spells, matrix, ulong = universe["spells"], universe["matrix"], universe["ulong"]
checks = U.spell_checks(spells)
print(json.dumps(checks, indent=1))
print("date values present that did not parse:", universe["unparsed"] or "none")
print(f"months {len(matrix)}, from {matrix.index[0]} to {matrix.index[-1]} | member-months {len(ulong):,} | {time.time() - t0:.0f} s")

## 2. The count check against the reference table

The universe's count in month t is set beside CRSP's count on its row for month t-1, which is the count on the same day, the last trading day of month t-1. The table by year is written with the same writer as Stock-Level Tracking's and compared with the reference table byte for byte, and by SHA-256.

In [ ]:
def against_reference(name, frame):
    """Write one table and compare its bytes with the reference table of the same name."""
    path = D.write_results(name, frame=frame)
    ours = path.read_bytes()
    theirs, where = D.reference_table("results/20260928/" + name)
    return {"table": name, "rows": int(len(frame)), "sha256": hashlib.sha256(ours).hexdigest()[:16],
            "reference_sha256": hashlib.sha256(theirs).hexdigest()[:16], "equal": ours == theirs, "reference_read_from": where}

counts = U.counts_against_crsp(matrix, universe["index_file"])
count_by_year = U.count_check_by_year(counts)
count_check = against_reference("universe_count_check_by_year.csv", count_by_year)
print(json.dumps(count_check, indent=1))

## 3. The monthly returns and the delisting counts

The monthly stock file is read for the universe's stocks, with the columns this study reads, and the delistings are read whole. The copied `monthly_panel` gives every member-month its total return with the delisting rule applied, its start-of-month capitalisation and how its delisting, if any, was treated. The counts by year are compared with the second reference table.

In [ ]:
t0 = time.time()
monthly, bad_monthly = D.read_table(root, "crsp_msf")
monthly = monthly[monthly["permno"].isin(matrix.columns)].reset_index(drop=True)
delistings, bad_delistings = D.read_table(root, "crsp_msedelist")
returns_panel = R.monthly_panel(ulong, monthly, delistings)
print(f"monthly rows for the universe's stocks {len(monthly):,} | member-months with a row in the panel {len(returns_panel):,} | {time.time() - t0:.0f} s")
print("date values present that did not parse:", {**bad_monthly, **bad_delistings} or "none")
delisting_check = against_reference("delistings_by_year.csv", R.delisting_by_year(returns_panel))
print(json.dumps(delisting_check, indent=1))

## 4. Gate G1 (a)

The first part of the gate is the member-months, their first and last month and their number of months, against the reference summary and the constants. The other two parts are the comparisons of sections 2 and 3. The verdict is written to `g1a.json`, and notebooks 03 to 05 stop if it is not met.

In [ ]:
reference_summary = json.loads(D.reference_table("results/20260928/universe_summary.json")[0])
members = {"member_months": int(len(ulong)), "first_month": str(matrix.index[0]), "last_month": str(matrix.index[-1]),
           "months": int(len(matrix))}
members_equal = (members["member_months"] == reference_summary["member_months"] == C.G1A_MEMBER_MONTHS
                 and members["first_month"] == reference_summary["first_month"] == C.G1A_FIRST_MONTH
                 and members["last_month"] == reference_summary["last_month"] == C.G1A_LAST_MONTH
                 and members["months"] == reference_summary["months"])
g1a = {"member_months": {**members, "reference": {k: reference_summary[k] for k in members}, "equal": members_equal},
       "count_check_by_year": count_check, "delistings_by_year": delisting_check,
       "g1a_met": bool(members_equal and count_check["equal"] and delisting_check["equal"])}
D.write_results("g1a.json", record=g1a)
print(json.dumps(members))
print("Gate G1 (a):", "met" if g1a["g1a_met"] else "not met", f"| member-months {members['member_months']:,} against {C.G1A_MEMBER_MONTHS:,}"
      f" | count check equal: {count_check['equal']} | delisting counts equal: {delisting_check['equal']}")

## 5. The panel's window

The panel keeps the member-months from January 1991 to December 2024. A member-month with no return and no delisting has no target and is counted. The counts are compared with the constants and judged by no gate.

In [ ]:
months = pd.PeriodIndex(returns_panel["month"], freq="M")
window = returns_panel[(months >= pd.Period(C.PANEL_FIRST_MONTH, "M")) & (months <= pd.Period(C.PANEL_LAST_MONTH, "M"))]
test = window[pd.PeriodIndex(window["month"], freq="M") >= pd.Period(C.TEST_FIRST_MONTH, "M")]
counts_window = {"panel_months": int(window["month"].nunique()), "panel_member_months": int(len(window)),
                 "member_months_without_return": int(window["return_missing"].sum()),
                 "test_months": int(test["month"].nunique()), "test_member_months": int(len(test))}
expected = {"panel_months": C.PANEL_MONTHS, "panel_member_months": C.PANEL_MEMBER_MONTHS,
            "member_months_without_return": C.PANEL_MEMBER_MONTHS_NO_RETURN, "test_months": C.TEST_MONTHS,
            "test_member_months": C.TEST_MEMBER_MONTHS}
print(pd.DataFrame({"result": counts_window, "constants": expected}).to_string())

## 6. What is committed

Four files go into `results/20260928/`, each an aggregate with no identifier of a security: the count check by year and the delisting counts by year, both equal to the reference tables when the gate is met, the gate's record `g1a.json`, and a summary of the counts printed here.

In [ ]:
summary = {"written_by": "notebooks/02_universe_and_returns.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
           "list": checks, **members, "monthly_rows_for_the_universe": int(len(monthly)), "panel_window": counts_window,
           "date_values_not_parsed": {**universe["unparsed"], **bad_monthly, **bad_delistings}}
p = D.write_results("universe_and_returns_summary.json", record=summary)
print("written: results/20260928/universe_count_check_by_year.csv, delistings_by_year.csv, g1a.json and", p.name)
results = pd.DataFrame([
    ("member-months, January 1926 to December 2024", f"{C.G1A_MEMBER_MONTHS:,}", f"{members['member_months']:,}", "met" if members_equal else "not met"),
    ("count check by year, byte for byte", "equal", "equal" if count_check["equal"] else "different", "met" if count_check["equal"] else "not met"),
    ("delisting counts by year, byte for byte", "equal", "equal" if delisting_check["equal"] else "different", "met" if delisting_check["equal"] else "not met"),
    ("member-months of the panel's window", f"{C.PANEL_MEMBER_MONTHS:,}", f"{counts_window['panel_member_months']:,}", "no verdict"),
    ("without a return or a delisting", str(C.PANEL_MEMBER_MONTHS_NO_RETURN), str(counts_window["member_months_without_return"]), "no verdict"),
    ("member-months of the test window", f"{C.TEST_MEMBER_MONTHS:,}", f"{counts_window['test_member_months']:,}", "no verdict"),
], columns=["expectation", "expected", "result", "verdict"])
print(results.to_string(index=False))
if not g1a["g1a_met"]:
    raise SystemExit("Gate G1 (a) is not met, so nothing after this notebook runs.")

## What was learned, and what could be wrong

- The table above sets each expectation beside its result. Gate G1 (a) is met only if all three of its parts are.
- Equal counts show that the copy is Stock-Level Tracking's layer on this extraction. They do not test the layer again: that study set the universe's count beside CRSP's count, and the universe's return beside the index's own return, in its notebooks 02 and 04.
- A count that agrees does not show that the right stocks are in the universe, only that their number is right. Stock-Level Tracking's comparison of returns is the finer test, and it stands for this study through the copy.
- The universe reads membership on the last trading day of the month before, so a stock that joins mid-month enters the universe the month after. Every characteristic in notebooks 03 and 04 is measured on that same day, so the panel uses nothing dated after the last trading day of the month before. Notebooks 04 and 05 give what this does not rule out: restated accounts and revised data.